In [1]:
# ===== PDF EXPORT: run this cell by itself AFTER "Run all" has finished =====
# Makes a PDF of this notebook with nothing cut off: long code lines and printed text wrap onto the next line,
# and wide tables and figures shrink to fit the page. This cell is left out of the PDF.
# Google Colab: reads the notebook straight from the browser, so it includes every output (no need to save first).
# Jupyter / VS Code: save the notebook first (Ctrl+S / Cmd+S), since it reads the saved file.
RUN_PDF_EXPORT = False                      # set to True, then run only this cell
NOTEBOOK_FILE = "01_data_cleaning.ipynb"                    # only used outside Colab
PDF_FILE = "01_data_cleaning.pdf"

if not RUN_PDF_EXPORT:
    print('PDF export is off. After "Run all" finishes, set RUN_PDF_EXPORT = True and run only this cell.')
else:
    import json, os, subprocess, sys, tempfile
    import nbformat
    from nbconvert import HTMLExporter

    # 1. Get the notebook with all of its outputs
    try:
        from google.colab import _message, files
        nb_dict = _message.blocking_request("get_ipynb", timeout_sec=120)["ipynb"]
        in_colab = True
    except ImportError:
        with open(NOTEBOOK_FILE, encoding="utf-8") as f:
            nb_dict = json.load(f)
        in_colab = False
    nb = nbformat.reads(json.dumps(nb_dict), as_version=4)
    nb.cells = [c for c in nb.cells if "RUN_PDF_EXPORT" not in "".join(c.source)]


    try:
        exporter = HTMLExporter(template_name="lab")
        html, _ = exporter.from_notebook_node(nb)
    except Exception:
        exporter = HTMLExporter(template_name="classic")
        html, _ = exporter.from_notebook_node(nb)
    print_css = """
    <style>
      @page { size: letter; margin: 0.5in; }
      body { margin: 0 !important; padding: 0 !important; font-size: 10.5pt; }
      .jp-Notebook, #notebook-container, .container { width: 100% !important; max-width: 100% !important;
                                                      padding: 0 !important; margin: 0 !important; box-shadow: none !important; }
      pre, code, .highlight pre, .jp-OutputArea-output pre {
          white-space: pre-wrap !important; word-break: normal !important; overflow-wrap: break-word !important;
          font-size: 8pt !important; }
      .jp-InputArea-editor, .jp-OutputArea, .jp-OutputArea-output, .jp-RenderedHTMLCommon, .output_area, .output_subarea {
          overflow: visible !important; max-width: 100% !important; }
      .jp-InputPrompt, .jp-OutputPrompt, .prompt { min-width: 0 !important; flex: 0 0 44px !important; font-size: 7pt !important; }
      table, .dataframe { font-size: 8pt !important; max-width: 100% !important; table-layout: auto !important; }
      th, td { white-space: normal !important; word-break: normal !important; overflow-wrap: normal !important;
               padding: 2px 5px !important; }   /* wrap only between words, so numbers never split */
      tr, img, svg { break-inside: avoid; }
      img { max-width: 100% !important; height: auto !important; }
      h1, h2, h3, h4 { break-after: avoid; }
      p, li { overflow-wrap: break-word; }
    </style>
    """
    html = html.replace("</head>", print_css + "</head>", 1)
    tmp = tempfile.mkdtemp()
    html_path = os.path.join(tmp, "notebook.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)


    try:
        import playwright
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "playwright"], check=True)
    chrome_path = os.environ.get("CHROME_PATH")
    if not chrome_path:
        install = [sys.executable, "-m", "playwright", "install"] + (["--with-deps"] if in_colab else []) + ["chromium"]
        subprocess.run(install, check=True, capture_output=True)
    pdf_path = os.path.abspath(PDF_FILE)
    script = f"""
from playwright.sync_api import sync_playwright
with sync_playwright() as p:
    browser = p.chromium.launch(executable_path={chrome_path!r} or None)
    page = browser.new_page()
    page.goto("file://" + {html_path!r}, wait_until="load")
    page.wait_for_timeout(3000)   # let MathJax and fonts finish
    page.pdf(path={pdf_path!r}, prefer_css_page_size=True, print_background=True)
    browser.close()
"""
    subprocess.run([sys.executable, "-c", script], check=True)
    print(f"Saved {pdf_path}")
    if in_colab:
        files.download(pdf_path)


PDF export is off. After "Run all" finishes, set RUN_PDF_EXPORT = True and run only this cell.


# INFO 2950 Final Project: Data Cleaning Notebook
Aiden Hwang, Anthony Ruiz, Youssef Sarhan

This is our data cleaning from Phase II, moved into its own notebook for Phase III. It reads our two raw files and saves the cleaned tables that our Phase III report imports (`classes_analysis_ready.csv`, `listings_analysis_ready.csv`,
`subject_semester_analysis_ready.csv`, `course_history.csv`, `trends_monthly_clean.csv`, `ai_labels_final.csv`, and `fn_labels_final.csv`). Steps 1–16 are the same as Phase II. Steps 14b, 14c, and 15b are new and respond to our Phase II feedback.

## To run this notebook:
please put `cornell_classes.csv`,       
`google_trends_raw.csv`,       
`ai_label_sheet_labeled.csv`,      
and `fn_label_sheet_labeled.csv`
(from the GitHub link
https://github.com/Aidennnnnnnnnn/2950-final-project/) in the same folder as the notebook.

## Additionally, please wait for all .csv files to finish loading before running, as `cornell_classes.csv` is a very large file. It took us around a minute to load them ourselves. Thank you!

### Data Cleaning
The initial files were massive, especially `cornell_meetings.csv` and `all_classes.jsonl`. Neither is used in this
notebook, because `cornell_classes.csv` contains the majority of the data we need.

In [2]:
# Load the libraries and the two raw files.
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)   # show full titles (no "...")
pd.set_option("display.max_columns", None)    # show every column
pd.set_option("display.width", 110)           # wide printed tables split into blocks that fit the page

raw = pd.read_csv("cornell_classes.csv", low_memory=False)
trends_raw = pd.read_csv("google_trends_raw.csv")

print(f"Raw course file: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"Semesters (rosters): {raw['roster'].nunique()}, subjects: {raw['subject'].nunique()}")
print(f"Google Trends file: {trends_raw.shape[0]} months, {trends_raw['date'].iloc[0]} to {trends_raw['date'].iloc[-1]}")
print("\nRows per term type:")
print(raw["semester"].value_counts().to_string())

Raw course file: 113,971 rows x 41 columns
Semesters (rosters): 50, subjects: 227
Google Trends file: 154 months, 2014-01-01 to 2026-10-01

Rows per term type:
semester
Fall      55390
Spring    52504
Summer     4938
Winter     1139


#### Step 1: Keep only Fall and Spring semesters
Summer and Winter sessions are short, offer few courses (about 400 and 100 per term, compared with about 4,300 in a
regular semester), and are mostly repeats of regular courses for students catching up. These are not honest
representations of Cornell's academic intent. Keeping only Fall and Spring also gives us an evenly spaced time series
(two terms per academic year).

In [3]:
# Keep only Fall and Spring semesters.
n_before = len(raw)
df = raw[raw["semester"].isin(["Fall", "Spring"])].copy()
print(f"Dropped {n_before - len(df):,} Summer/Winter rows; {len(df):,} Fall/Spring rows remain")
print(f"Semesters kept: {df['roster'].nunique()} ({df.sort_values('strm')['roster'].iloc[0]} to "
      f"{df.sort_values('strm')['roster'].iloc[-1]})")

Dropped 6,077 Summer/Winter rows; 107,894 Fall/Spring rows remain
Semesters kept: 25 (FA14 to FA26)


#### Step 2: Look at every column before deciding what to keep
The table below shows, for each of the 41 raw columns, how much is missing and an example value. We use it to justify
each keep / drop decision.

In [4]:
# Check every column before deciding what to keep: how full it is and how many distinct values it has.
summary = pd.DataFrame({
    "% missing": (df.isna().mean() * 100).round(1),
    "distinct values": df.nunique(),
    "example": df.apply(lambda col: col.dropna().iloc[0] if col.notna().any() else None),
})
summary["example"] = summary["example"].astype(str).str[:45]
summary

,% missing,distinct values,example
roster,0.0,25,FA14
year,0.0,13,2014
semester,0.0,2,Fall
queried_subject,0.0,225,AAS
catalog_dttm,12.5,20,2015-06-11T18:21:23-0400
strm,0.0,25,2573
crseId,0.0,13859,352628
crseOfferNbr,0.0,13,2
subject,0.0,225,AAS
catalogNbr,0.0,4591,2130


We decided to keep a column only if it tells us when a class was offered, which department and level it belongs to, which
course it is, or what it is about. Everything else only eats up unnecessary space.

| Column(s) | Keep? | Why |
|---|---|---|
| `roster`, `year`, `semester` | Keep | When the class was offered. Time is the backbone of our question, and it is how we join Google Trends. |
| `strm` | Keep | Cornell's numeric term code. It increases every term, so it sorts semesters correctly (text codes like `FA14` and `SP15` don't sort in time order). |
| `subject` | Keep | The department code (e.g., `CS`). We group these into disciplines for Analysis 2. |
| `catalogNbr` | Keep | The course number. Its first digit gives the course level for Analysis 1. |
| `crseId` | Keep | Cornell's permanent ID for a course. It is shared by cross-listings (so we can count a class once) and stays the same when a course is renumbered or renamed (Step 10). |
| `crseOfferNbr` | Keep | Which listing of a cross-listed course this row is. Offer number 1 is the registrar's main listing, which we use to pick one row per class (Step 9). |
| `titleLong`, `description` | Keep | The text we search for AI keywords. |
| `acadCareer` | Keep | The registrar's own label for undergraduate (UG) or graduate/professional (GR, LA = law, VM = vet, GM = MBA) courses. We use it to check our number-based level (Step 7). |
| `acadGroup` | Keep | The college that owns the course. We use it to check our definition of "Computing" (Step 5). |
| `queried_subject` | Drop | The subject our script asked for. It always equals `subject` (checked below), so it is a duplicate. |
| `titleShort` | Drop | A shortened transcript version of `titleLong` (e.g., "Intro to Hebrew Bible"). Searching the full title is enough, and abbreviations could hide keywords. |
| `catalog_dttm` | Drop | When that semester's catalog was published. It describes the catalog file, not the course, and is empty from Fall 2025 on. |
| `catalogWhenOffered`, `lastTermsOffered` | Drop | Text like "Fall." or "2025FA, 2024FA". We already know exactly when each class ran from `roster`. |
| `catalogDistr`, `catalogBreadth`, `catalogSatisfiesReq`, `catalogAttribute`, `catalogCourseSubfield`, `catalogLang` | Drop | Which requirements a course satisfies (e.g., "(HA-AS)"). They describe degree rules, not course content, and are 78–98% missing. |
| `catalogPrereqCoreq`, `catalogPrereq`, `catalogCoreq`, `catalogForbiddenOverlaps` | Drop | Prerequisites and overlap rules. Interesting (does an AI course outside CS require CS classes?), but 71–100% missing, so any result would rest on a small, unusual subset. |
| `catalogEnrollmentPriority`, `catalogPermission`, `catalogFee`, `catalogComments` | Drop | Enrollment logistics (who gets priority, lab fees, "credit hours TBA"). Not about content. We also don't search `catalogComments` for keywords, because it often has administrative notes rather than course topics. |
| `catalogOutcomes`, `exploreCriteriaIds`, `catalogName`, `catalogNextOffered`, `catalogCredits`, `catalogGradeOption`, `catalogLocation`, `catalogCrosslisted`, `catalogCrosslistings`, `catalogComeets` | Drop | 100% empty. These were lists in the API response, which a flat CSV can't store (see Data Collection). |

We also rename the columns we keep to snake_case (e.g., `catalogNbr` → `catalog_nbr`) so the code reads consistently, and
rename `acadGroup` to `college` and `titleLong` to `title` because those names say what the columns mean.

In [5]:
# Confirm the two "duplicate" columns really are duplicates before dropping them.
print("Rows where subject != queried_subject:", (df["subject"] != df["queried_subject"]).sum())
print("\nExample: titleShort is the abbreviated transcript title, titleLong is the full title")
print(df[["titleShort", "titleLong"]].drop_duplicates().sample(3, random_state=2950).to_string(index=False))

KEEP = ["roster", "strm", "year", "semester", "subject", "catalogNbr", "crseId", "crseOfferNbr",
        "titleLong", "description", "acadCareer", "acadGroup"]
dropped = [c for c in df.columns if c not in KEEP]
df = df[KEEP].rename(columns={"catalogNbr": "catalog_nbr", "crseId": "crse_id", "crseOfferNbr": "crse_offer_nbr",
                              "titleLong": "title", "acadCareer": "acad_career", "acadGroup": "college"})
print(f"\nKept {len(KEEP)} columns, dropped {len(dropped)}:")
print(dropped)

Rows where subject != queried_subject: 0

Example: titleShort is the abbreviated transcript title, titleLong is the full title
                    titleShort                         titleLong
   Turbomachinery Applications       Turbomachinery Applications
         Intro to Hebrew Bible  Introduction to the Hebrew Bible
Making and Selling Creative Id Making and Selling Creative Ideas

Kept 12 columns, dropped 29:
['queried_subject', 'catalog_dttm', 'titleShort', 'catalogBreadth', 'catalogDistr', 'catalogEnrollmentPriority', 'catalogLang', 'catalogForbiddenOverlaps', 'catalogWhenOffered', 'catalogComments', 'catalogPrereqCoreq', 'catalogPrereq', 'catalogCoreq', 'catalogFee', 'catalogSatisfiesReq', 'catalogPermission', 'catalogCourseSubfield', 'catalogOutcomes', 'lastTermsOffered', 'catalogAttribute', 'exploreCriteriaIds', 'catalogName', 'catalogNextOffered', 'catalogCredits', 'catalogGradeOption', 'catalogLocation', 'catalogCrosslisted', 'catalogCrosslistings', 'catalogComeets']


#### Step 3: Missing values, extra spaces, duplicate rows
* **Missing identifiers:** A row without a semester, subject, number, course ID, or title can't be placed in time,
  department, or level, so we drop that row.
* **Missing descriptions:** We actually keep these rows, because the title alone can still show AI content. Dropping them
  would remove legitimate courses and could bias the counts, as missing descriptions are more common in the early
  semesters (7.3% in Fall 2014 vs. 1.0% in Fall 2026, shown below). We fill the gap with an empty string so text searches
  work and add a flag, `has_description`, so we know these rows could only be classified by title.
* **Extra spaces:** We strip unnecessary spaces at the start and end of names.
* **Duplicates:** We check for any duplicates, since we can't manually check tens of thousands of rows.

In [6]:
n_start = len(df)

# Rows missing an identifier can't be placed in time, department, or level.
df = df.dropna(subset=["roster", "subject", "catalog_nbr", "crse_id", "title"])
print(f"Dropped {n_start - len(df)} rows missing an identifier or title")

# Keep rows without a description, but flag them.
df["description"] = df["description"].fillna("")
for col in ["subject", "title", "description"]:
    df[col] = df[col].str.strip()
df["has_description"] = (df["description"] != "").astype(int)
print(f"{(df['has_description'] == 0).sum():,} listings ({(df['has_description'] == 0).mean():.1%}) have no description")

# The same subject + number twice in one semester would be a scraping error.
n_dups = df.duplicated(subset=["roster", "subject", "catalog_nbr"]).sum()
df = df.drop_duplicates(subset=["roster", "subject", "catalog_nbr"])
print(f"Removed {n_dups} duplicate subject-number-semester rows")

print("\nShare of listings with no description, by semester (%):")
print((100 * (1 - df.groupby("roster", sort=False)["has_description"].mean())).round(1).to_frame("% missing").T)

Dropped 1 rows missing an identifier or title
3,795 listings (3.5%) have no description
Removed 0 duplicate subject-number-semester rows

Share of listings with no description, by semester (%):
roster     FA14  SP15  FA15  SP16  FA16  SP17  FA17  SP18  FA18  SP19  FA19  SP20  FA20  SP21  FA21  SP22  \
% missing   7.3   7.7   4.8   5.9   3.1   4.2   2.9   4.7   3.8   4.9   3.1   4.0   2.4   4.1   3.0   4.1   

roster     FA22  SP23  FA23  SP24  FA24  SP25  FA25  SP26  FA26  
% missing   2.4   3.9   2.1   2.9   1.0   2.5   1.5   1.5   1.0  


#### Step 4: Create the AI variables
* **`is_ai`** (1 or 0): our main AI/ML measure. We lowercase the title + description and search for whole-word matches
  (the `\b` word boundaries in the regular expression) of these specific terms: *machine learning, artificial
  intelligence, neural network(s), deep learning, computer vision, natural language processing, reinforcement learning,
  large language model(s), generative AI*
  * We chose these terms because each one names an AI/ML subject or field. A course description that uses one is most
    likely discussing AI/ML.
  * We did not use shorter words such as "learning", "neural", "data", or "algorithm" because they have a very high
    chance of matching with other unrelated courses (education, neuroscience, statistics, etc).
* **`ai_keyword`**: the first keyword that matched, saved so we can check our flags (Step 14).
* **`mentions_ai`** (1 or 0): a broader measure. It is 1 if `is_ai` is 1 or the text contains the abbreviation "AI" or
  "ChatGPT".
  * We added this because many courses outside computing mention AI as a topic ("AI and digital marketing", "AI
    governance") without using technical terms. Our research question is about how AI spreads across disciplines, so we
    want to see this too. We keep it separate from `is_ai` because mentioning AI is separate from actually studying it.
  * "AI" must be capitalized because lowercase "ai" also matches unrelated words, like "T'ai Chi" (PE 1380) and the
    artist Ai Weiwei (COML 2030), shown in the output below.

In [7]:
# Two AI variables, from the course title + description.
CORE_KEYWORDS = [
    "machine learning", "artificial intelligence", "neural networks?", "deep learning",
    "computer vision", "natural language processing", "reinforcement learning",
    "large language models?", "generative ai",
]
core_regex = r"\b(" + "|".join(CORE_KEYWORDS) + r")\b"

text = df["title"] + " " + df["description"]
df["ai_keyword"] = text.str.lower().str.extract(core_regex, expand=False).str.replace(r"s$", "", regex=True)
df["is_ai"] = df["ai_keyword"].notna().astype(int)

# Broader: also counts the bare abbreviation "AI" (capital letters only) and "ChatGPT".
broad_extra = text.str.contains(r"\bAI\b|ChatGPT", regex=True)
df["mentions_ai"] = ((df["is_ai"] == 1) | broad_extra).astype(int)

print(f"is_ai = 1:       {df['is_ai'].sum():,} listings ({df['is_ai'].mean():.2%})")
print(f"mentions_ai = 1: {df['mentions_ai'].sum():,} listings ({df['mentions_ai'].mean():.2%})")
print(f"Listings that only the broad measure catches: {((df['mentions_ai'] == 1) & (df['is_ai'] == 0)).sum()}")

# Why case-sensitive: lowercase "ai" also matches "T'ai Chi" (PE 1380) and the artist Ai Weiwei (COML 2030).
lower_only = text.str.contains(r"(?i)\bai\b") & ~text.str.contains(r"\bAI\b")
print("\nTitles matched by lowercase 'ai' but not capital 'AI':")
for _, row in df[lower_only].drop_duplicates(["subject", "catalog_nbr"]).iterrows():
    t = row["title"] + " " + row["description"]
    hit = t.lower().find("ai ")
    print(f"{row['subject']} {row['catalog_nbr']}: ...{t[max(0, hit - 25):hit + 25]}...")

is_ai = 1:       1,671 listings (1.55%)
mentions_ai = 1: 1,818 listings (1.69%)
Listings that only the broad measure catches: 147

Titles matched by lowercase 'ai' but not capital 'AI':
PE 1380: ...Tai Chi Chuan Fundamentall...
COML 2030: ...ond Williams, Teju Cole, Ai WeiWei, Abderrahmane S...


#### Step 5: Grouping subjects by discipline and removing non-departmental subjects
Our research question asks about disciplines, but 225 subject codes are too many to compare. We group them into five
discipline groups, assigning each subject by what it teaches. This is how we can utilize the purest stream of data from
Cornell while not being excessively granular in our analysis.

| Discipline | Examples | Count |
|---|---|---|
| Computing | Cornell Bowers CIS: computer science, information science, statistics and data science | 3 |
| Engineering & Math | engineering departments (ECE, ORIE, MAE, CEE, ...), MATH, biometry (BTRY), Cornell Tech (TECH) | 21 |
| Natural & Life Sciences | biology, chemistry, physics, astronomy, plant and animal sciences, nutrition, vet medicine, entomology | 41 |
| Social Sciences, Business & Law | economics, government, psychology, sociology, linguistics, cognitive science, business schools, ILR, law, public policy | 48 |
| Humanities & Arts | history, philosophy, English, all languages and literatures, art, architecture, music, design | 95 |

CS, INFO, and STSCI make up the Cornell Bowers College of Computing and Information Science, where we expect AI to start
appearing. Bowers CIS (`CI`) appears on exactly these three subjects (below). Engineering also teaches a lot of AI/ML, so
we create a second version, `is_computing_broad`, that also counts Engineering & Math as computing. Analysis 2 checks
whether our conclusions depend on that choice.

**Why we remove "non-departmental" subjects (5,656 listings):** Some subject codes are not academic departments:
physical education (PE), ROTC (MILS, NAVS, AIRS), college-wide advising, leadership, and honors seminars (ALS, HE, AS,
LEAD, COLLS, UNILWYL), study abroad (OVST), graduate school placeholders (GRAD), and English-language support (ELSO). The
full list, with row counts, is printed below. They can't tell us whether a discipline adopted AI, and PE alone adds 3,709
introductory-level listings that would make the Intro-level course group of AI look smaller for unrelated reasons. None of
them has an `is_ai` course (8 have `mentions_ai`), so removing them can't hide any AI courses.

Generally, since we are working with a data set this large, we wanted to take any avenue we could think of that would
eliminate any unnecessary bloat data.

In [8]:
# Group the 225 subject codes into broad disciplines.
DISCIPLINE_SUBJECTS = {
    "Computing": ["CS", "INFO", "STSCI"],
    "Engineering & Math": [
        "AEP", "BEE", "BME", "CEE", "CHEME", "ECE", "ENGRC", "ENGRD", "ENGRG", "ENGRI", "ENGRP",
        "ENMGT", "MAE", "MSE", "NSE", "ORIE", "SYSEN", "MATH", "BTRY", "TECH", "TECHIE"],
    "Natural & Life Sciences": [
        "ANSC", "ASTRO", "BIOAP", "BIOCB", "BIOEE", "BIOG", "BIOMG", "BIOMI", "BIOMS", "BIONB",
        "BIOPL", "CHEM", "CSS", "EAS", "ENTOM", "ENVS", "ESS", "FDSC", "HORT", "NS", "NTRES",
        "PHYS", "PLBIO", "PLBR", "PLBRG", "PLHRT", "PLPA", "PLPPM", "PLSCI", "PLSCS", "SEA",
        "SNES", "TOX", "VETCS", "VETMI", "VETMM", "VIEN", "VTBMS", "VTMED", "VTPEH", "VTPMD"],
    "Social Sciences, Business & Law": [
        "AEM", "ANTHR", "BANA", "BPRE", "BSOC", "CAPS", "COGST", "COMM", "CRP", "DEA", "DSOC",
        "ECON", "EDUC", "GDEV", "GOVT", "HADM", "HD", "IARD", "ILRGL", "ILRHR", "ILRIC", "ILRID",
        "ILRLE", "ILRLR", "ILROB", "ILRST", "LAW", "LEGAL", "LING", "MGMT", "NACCT", "NBA",
        "NBAB", "NBAE", "NBAT", "NBAY", "NCC", "NCCB", "NCCY", "NMI", "NRE", "PADM", "PAM",
        "PSYCH", "PUBPOL", "REAL", "SOC", "STS"],
    "Humanities & Arts": [
        "AAP", "AAS", "AIIS", "AIS", "AKKAD", "AMST", "ARAB", "ARCH", "ARKEO", "ART", "ARTH",
        "ASIAN", "ASL", "ASRC", "BCS", "BENGL", "BURM", "CATAL", "CHERO", "CHIN", "CHLIT",
        "CLASS", "COML", "CZECH", "DESIGN", "DUTCH", "ENGL", "FGSS", "FINN", "FREN", "FSAD",
        "GERST", "GREEK", "HEBRW", "HIERO", "HINDI", "HIST", "HUNGR", "INDO", "ITAL", "JAPAN",
        "JPLIT", "JUARA", "JWST", "KANAD", "KHMER", "KOREA", "KRLIT", "LA", "LATA", "LATIN",
        "LGBT", "LSP", "MEDVL", "MOGRK", "MUSIC", "NEPAL", "NES", "PALI", "PERSN", "PHIL", "PMA",
        "POLSH", "PORT", "PUNJB", "QUECH", "RELST", "ROMAN", "ROMS", "RUSSA", "RUSSL", "SANSK",
        "SHUM", "SINHA", "SNLIT", "SPAN", "SUMER", "SWAHL", "SWED", "TAG", "TAMIL", "TELU",
        "THAI", "TIBET", "TURK", "TWI", "UKRAN", "URDU", "VIET", "VISST", "WOLOF", "WRIT",
        "YIDSH", "YORUB", "ZULU"],
    "Non-departmental": [
        "PE", "MILS", "NAVS", "AIRS", "UNIV", "UNILWYL", "GRAD", "OVST", "ENINT", "COLLS", "AS",
        "IM", "ELSO", "HE", "ALS", "AGSCI", "LEAD"],
}
DISCIPLINE = {subj: group for group, subjects in DISCIPLINE_SUBJECTS.items() for subj in subjects}

unmapped = sorted(set(df["subject"]) - set(DISCIPLINE))
assert not unmapped, f"Subjects without a discipline: {unmapped}"
df["discipline"] = df["subject"].map(DISCIPLINE)

# Check our "Computing" choice against the registrar: which subjects are ever owned by Bowers CIS (college code CI)?
print("Subjects ever listed under college 'CI' (Bowers CIS):", sorted(df.loc[df["college"] == "CI", "subject"].unique()))

nondept = df[df["discipline"] == "Non-departmental"]
print(f"\nNon-departmental listings removed: {len(nondept):,} "
      f"({nondept['is_ai'].sum()} with is_ai = 1, {nondept['mentions_ai'].sum()} with mentions_ai = 1)")
print(nondept.groupby("subject").size().sort_values(ascending=False).to_string())
df = df[df["discipline"] != "Non-departmental"].copy()

df["is_computing"] = (df["discipline"] == "Computing").astype(int)
df["is_computing_broad"] = df["discipline"].isin(["Computing", "Engineering & Math"]).astype(int)
print(f"\nListings remaining: {len(df):,} in {df['subject'].nunique()} subjects")
df.groupby("discipline").agg(subjects=("subject", "nunique"), listings=("subject", "size"),
                             ai_listings=("is_ai", "sum"))

Subjects ever listed under college 'CI' (Bowers CIS): ['CS', 'INFO', 'STSCI']

Non-departmental listings removed: 5,656 (0 with is_ai = 1, 8 with mentions_ai = 1)
subject
PE         3709
ALS         357
HE          254
NAVS        234
MILS        216
AIRS        175
ELSO        159
UNILWYL     157
GRAD         94
COLLS        83
LEAD         75
AS           63
IM           44
AGSCI        28
UNIV          4
ENINT         2
OVST          2

Listings remaining: 102,237 in 208 subjects


,subjects,listings,ai_listings
discipline,,,
Computing,3,4493,742
Engineering & Math,21,14688,421
Humanities & Arts,95,32800,43
Natural & Life Sciences,41,17611,107
"Social Sciences, Business & Law",48,32645,358


#### Step 6: Remove placeholder courses (independent study, thesis, internships)
Some catalog entries aren't real courses, which was surprising. They are placeholders such as "Independent Study,"
"Thesis Research," or "Internship." Others are "Teaching Experience" (TA credit courses), "Directed Readings," or "Special
Projects." They are listed every semester, have overly generic descriptions, and are usually upper-level or graduate. If
we kept those 8,047 listings, they would add many non-AI rows to those levels and make AI shares look smaller for
unrelated reasons.

We find them with a title pattern that we wrote in code and then confirm that none of them is flagged as `is_ai`. This
removes noise while not colliding with our adjacent cleaning steps.

In [9]:
placeholder_regex = (
    r"(?i)\b(?:independent (?:study|research|readings?)|thesis|dissertation|internship|"
    r"supervised (?:research|readings?)|undergraduate research|teaching experience|"
    r"directed (?:readings?|study|studies)|special projects?)\b|^(?:graduate research|research in\b|research projects?\b)"
)
df["is_placeholder"] = df["title"].str.contains(placeholder_regex, regex=True)

print(f"Placeholder listings: {df['is_placeholder'].sum():,} ({df['is_placeholder'].mean():.1%} of listings)")
print("Placeholders with is_ai = 1:", int(df.loc[df["is_placeholder"], "is_ai"].sum()))
removed = df[df["is_placeholder"]]
df = df[~df["is_placeholder"]].drop(columns="is_placeholder")
print(f"Listings after this step: {len(df):,}")
removed["title"].value_counts().head(8).to_frame("listings removed")

Placeholder listings: 8,047 (7.9% of listings)
Placeholders with is_ai = 1: 0
Listings after this step: 94,190


,listings removed
title,
Independent Study,463
Master's Level Thesis Research,209
Undergraduate Research,204
Teaching Experience,179
Graduate Thesis,176
Doctoral-Level Thesis Research,169
Directed Readings,163
Thesis Research,142


#### Step 7: Course level, checked against the registrar
* `course_level`, from the first digit of `catalog_nbr`:
  * **Intro** = 1000–2999 (introductory undergraduate courses)
  * **Upper** = 3000–4999 (upper-level undergraduate courses)
  * **Grad** = 5000 and above (graduate / professional courses)
* `is_intro`, `is_upper` (1 or 0): the same information as two indicators for the regression, with Grad as the
  comparison group.

**Checking our cutoffs.** We compared the course numbers with the registrar's own label, `acad_career`. The table below
shows they align very well: every undergraduate (UG) course is numbered below 5000 except 73, and every graduate or
professional course (GR, LA, VM, GM) is numbered 5000+ except 1. So the number is a reliable measure of level.

In [10]:
def assign_course_level(num):
    if 1000 <= num < 3000:
        return "Intro"
    if 3000 <= num < 5000:
        return "Upper"
    if num >= 5000:
        return "Grad"
    return np.nan

df["catalog_nbr"] = df["catalog_nbr"].astype(int)
df["course_level"] = df["catalog_nbr"].apply(assign_course_level)
print("Listings below 1000 (no degree level):", df["course_level"].isna().sum())
df = df.dropna(subset=["course_level"])
df["is_intro"] = (df["course_level"] == "Intro").astype(int)
df["is_upper"] = (df["course_level"] == "Upper").astype(int)

# Check the number-based level against the registrar's own label (acad_career).
print("\nOur level (columns) vs. registrar's academic career (rows):")
# Count rows in each (career, level) pair, then spread levels into columns (unstack, Lecture 6).
level_check = df.groupby(["acad_career", "course_level"]).size().unstack(fill_value=0)
level_check["All"] = level_check.sum(axis=1)   # row totals
level_check.loc["All"] = level_check.sum()     # column totals
level_check[["Intro", "Upper", "Grad", "All"]]

Listings below 1000 (no degree level): 0

Our level (columns) vs. registrar's academic career (rows):


course_level,Intro,Upper,Grad,All
acad_career,,,,
GM,0,0,2771,2771
GR,0,1,30613,30614
LA,0,0,3492,3492
UG,21426,33759,73,55258
VM,0,0,2055,2055
All,21426,33760,39004,94190


#### Step 8: Time variables
* `term_index`: counts semesters from Fall 2014 (0) to Fall 2026 (24). We build it from `strm` so it is always in the
  right order.
* `years_since_start`: `term_index / 2`, so regression slopes mean "change per year", and 0 means Fall 2014.
* `academic_year`: pairs each Fall with the following Spring (e.g., "2014–15"). We use it for plots so every point has
  two semesters. The last one, 2026–27, only has Fall 2026 so far.

In [11]:
# Time variables. strm is Cornell's term code
term_order = sorted(df["strm"].unique())
df["term_index"] = df["strm"].map({s: i for i, s in enumerate(term_order)})
df["years_since_start"] = df["term_index"] / 2
ay_start = np.where(df["semester"] == "Fall", df["year"], df["year"] - 1)
df["academic_year"] = [f"{y}–{str(y + 1)[-2:]}" for y in ay_start]

print(df.groupby(["term_index", "roster", "academic_year"]).size().rename("listings").reset_index()
        .iloc[[0, 1, 2, -3, -2, -1]].to_string(index=False))

 term_index roster academic_year  listings
          0   FA14       2014–15      3385
          1   SP15       2014–15      3464
          2   FA15       2015–16      3461
         22   FA25       2025–26      3936
         23   SP26       2025–26      4089
         24   FA26       2026–27      3941


#### Step 9: Count cross-listed classes once
Many classes are cross-listed: one class, listed under several departments. For example, CS / INFO and CS / ECE / ORIE
share AI courses. In the data, all listings of one class share the same `roster` and `crse_id`. A third of all listings
belong to a cross-listed class, so this matters.

Whether to count a cross-listed class once or once per department depends on what each analysis asks, so we build two
tables:
* **`classes`** (one row per class per semester), used in Analysis 1 (course level). The question is how many classes at
  each level cover AI. Counting Networks as CS 2850, ECON 2040, INFO 2040, and SOC 2090 would turn one class into four.
  When a class has several listings, we keep the registrar's main listing (offer number 1). In a few dozen classes,
  listing number 1 was a non-departmental or placeholder row we already removed, so we keep the lowest-numbered listing
  that is left. This removes 289 duplicate AI listings.
* **`listings`** (one row per department listing per semester), used in Analysis 2 (disciplines). The question is which
  departments offer AI to their students. A department that cross-lists an AI class has chosen to put it in its own
  catalog, so it gets credit. We flag every cross-listed row with `is_crosslisted` so we can test later whether growth
  outside computing comes only from cross-listing.

In [12]:
# A cross-listed class appears once per department, all sharing (roster, crse_id).
listings_per_class = df.groupby(["roster", "crse_id"]).size().rename("n_listings").reset_index()
df = df.merge(listings_per_class, on=["roster", "crse_id"], how="left", validate="many_to_one")
df["is_crosslisted"] = (df["n_listings"] > 1).astype(int)
df = df.drop(columns="n_listings")

# Analysis 1 table
df = df.sort_values(["strm", "crse_id", "crse_offer_nbr"])
classes = df.drop_duplicates(subset=["roster", "crse_id"], keep="first").copy()
listings = df.copy()

print(f"Listings that belong to a cross-listed class: {df['is_crosslisted'].sum():,} ({df['is_crosslisted'].mean():.1%})")
print(f"listings (one row per department listing): {len(listings):,} rows, {listings['is_ai'].sum():,} with is_ai = 1")
print(f"classes (one row per class per semester): {len(classes):,} rows, {classes['is_ai'].sum():,} with is_ai = 1")
print(f"\nAI listings removed by counting each class once: {listings['is_ai'].sum() - classes['is_ai'].sum()}")

pairs = (listings[(listings["is_crosslisted"] == 1) & (listings["is_ai"] == 1)]
         .groupby(["roster", "crse_id"])["subject"].apply(lambda s: " / ".join(sorted(s))))
print("\nMost common cross-listed AI classes (class-semesters):")
pairs.value_counts().head(8).to_frame("class-semesters")

Listings that belong to a cross-listed class: 31,888 (33.9%)
listings (one row per department listing): 94,190 rows, 1,671 with is_ai = 1
classes (one row per class per semester): 75,082 rows, 1,382 with is_ai = 1

AI listings removed by counting each class once: 289

Most common cross-listed AI classes (class-semesters):


,class-semesters
subject,
CS / INFO,31
CHEME / SYSEN,17
INFO / LAW,14
COGST / CS / LING,12
CS / ECON / INFO / SOC,11
CS / ECE / ORIE,10
CEE / ENMGT,8
CRP / DESIGN,8


#### Step 10: Follow a course when its number or name changes
Our pre-Phase II feedback pointed out that courses get renumbered and renamed, which could make one course look like
several courses. Cornell's `crse_id` actually follows a course through those changes, as seen here:
* **Machine Learning** (`crse_id` 358592) was CS 4780 under three different titles, then was renumbered to CS 3780 in Fall
  2024.
* **Foundations of Artificial Intelligence** (`crse_id` 358585) was CS 4700, then became CS 3700 with a new title in Fall
  2024.

Because `crse_id` stays the same, it is our course identity: anything we count "per course" (Step 13) uses `crse_id`, so
a renamed or renumbered course is still one course. These two examples are also interesting for our question. A course
as important as a core AI course going from the 4000s to the 3000s is an indication of some of the "trickle-down" we are
looking for, even though both numbers are in our Upper level.

In [13]:
# Does crse_id follow a course when its number or title changes? Look at two well-known AI courses.
for subj, nbr in [("CS", 4780), ("CS", 4700)]:
    cid = listings.loc[(listings["subject"] == subj) & (listings["catalog_nbr"] == nbr), "crse_id"].iloc[0]
    history = (listings[(listings["crse_id"] == cid) & (listings["subject"] == subj)]
               .groupby(["subject", "catalog_nbr", "title"], sort=False)["roster"]
               .agg(first_semester="first", last_semester="last", semesters="size"))
    print(f"crse_id {cid}:")
    display(history)

ids = listings.groupby("crse_id").agg(titles=("title", "nunique"), levels=("course_level", "nunique"),
                                      ever_ai=("is_ai", "max"))
# Renumbered = more than one number within the SAME subject (different subjects are just cross-listings).
renumbered = listings.groupby(["crse_id", "subject"])["catalog_nbr"].nunique().gt(1).groupby("crse_id").any()
ids["renumbered"] = renumbered
print(f"Course identities (crse_id): {len(ids):,}")
print(f"  renumbered within the same subject: {ids['renumbered'].sum():,} "
      f"({ids.loc[ids['ever_ai'] == 1, 'renumbered'].sum()} of them ever AI)")
print(f"  offered under more than one title (renamed): {(ids['titles'] > 1).sum():,}")
print(f"  whose number moved between levels: {(ids['levels'] > 1).sum():,} "
      f"({(ids['levels'] > 1)[ids['ever_ai'] == 1].sum()} of them ever AI)")

crse_id 358592:


first_semester last_semester  semesters
subject catalog_nbr title                                                                           
CS      4780        Machine Learning                                   FA14          FA14          1
                    Machine Learning for Intelligent Systems           FA15          FA19          5
                    Introduction to Machine Learning                   FA20          SP24          8
        3780        Introduction to Machine Learning                   FA24          FA26          5

crse_id 358585:


first_semester last_semester  semesters
subject catalog_nbr title                                                                                  
CS      4700        Foundations of Artificial Intelligence                    FA14          SP24         18
        3700        Foundations of AI Reasoning and Decision-Making           FA24          FA26          5

Course identities (crse_id): 12,876
  renumbered within the same subject: 303 (13 of them ever AI)
  offered under more than one title (renamed): 1,767
  whose number moved between levels: 97 (3 of them ever AI)


#### Step 11: Merge renamed departments
Departments get renamed as well, as seen in how when the Brooks school of Public Policy opened, Policy Analysis and
Management (PAM) courses converted to PUBPOL, and the plant sciences merged several course codes into PLSCI. If we left
these intact, one department would look like two in Analysis 2.

We found renames from the data: for every subject code that stops before Fall 2026, we check where its courses
(`crse_id`) are listed. If a significant portion (we said 40%) of the old code's courses (and at least 5) continue under
one new code, we see it as renamed. We follow chains (like CSS → PLSCS → PLSCI) and check that no rename crosses
disciplines.

**Why 40% and not 30% or 50% or 100%:** when a department is renamed, it usually retires some old courses at the same
time, so even a real rename doesn't carry over every course. Renames such as PAM → PUBPOL (65%) and DSOC → GDEV (56%),
are above 40%. **Why at least 5 courses:** we set this floor so that a code with only two or three courses can't count as
a rename, cause that would be closer to guessing.

`department`: the current code for each subject after merging renames. 208 subject codes become 193 departments. We use
`department` to group standard errors in Analysis 2, so a renamed department counts as one group.

In [14]:
# For each subject code that stops before Fall 2026, find where its courses (crse_id) went next.
last_strm = listings["strm"].max()
span = listings.groupby("subject")["strm"].agg(["min", "max"])
rows = []
for old in span.index[span["max"] < last_strm]:
    old_ids = set(listings.loc[listings["subject"] == old, "crse_id"])
    later = listings[listings["crse_id"].isin(old_ids) & (listings["strm"] > span.loc[old, "max"])
                     & (listings["subject"] != old)]
    if later.empty:
        continue
    moved = later.groupby("subject")["crse_id"].nunique().sort_values(ascending=False)
    rows.append({"old_code": old, "successor": moved.index[0], "courses_moved": moved.iloc[0],
                 "old_courses": len(old_ids)})
moves = pd.DataFrame(rows)
moves["share_moved"] = (moves["courses_moved"] / moves["old_courses"]).round(2)

# Treat it as a rename only if at least 40% of the old code's courses (and >= 5) moved to one successor.
renames = moves[(moves["share_moved"] >= 0.4) & (moves["courses_moved"] >= 5)]
RENAME = dict(zip(renames["old_code"], renames["successor"]))

def current_code(subj):
    while subj in RENAME:                 # follow chains such as CSS -> PLSCS -> PLSCI
        subj = RENAME[subj]
    return subj

listings["department"] = listings["subject"].map(current_code)
classes["department"] = classes["subject"].map(current_code)
assert (listings.groupby("department")["discipline"].nunique() == 1).all(), "a rename crossed disciplines"
print(f"Subject codes: {listings['subject'].nunique()} -> departments after merging renames: "
      f"{listings['department'].nunique()}\n")
renames.sort_values("courses_moved", ascending=False).assign(
    final_department=lambda d: d["old_code"].map(current_code)).reset_index(drop=True)

Subject codes: 208 -> departments after merging renames: 193



,old_code,successor,courses_moved,old_courses,share_moved,final_department
0,PAM,PUBPOL,136,209,0.65,PUBPOL
1,ILRLR,ILRGL,65,133,0.49,ILRGL
2,PADM,PUBPOL,61,105,0.58,PUBPOL
3,DSOC,GDEV,59,105,0.56,GDEV
4,CSS,PLSCS,40,42,0.95,PLSCI
5,HORT,PLHRT,40,51,0.78,PLSCI
6,PLBIO,PLSCI,39,54,0.72,PLSCI
7,PLSCS,PLSCI,36,60,0.60,PLSCI
8,PLHRT,PLSCI,29,65,0.45,PLSCI
9,BIOPL,PLBIO,25,33,0.76,PLSCI


#### Step 12: Why we don't merge courses by title
We can't do that because they aren't the same courses: Some classes might have a similarly named course in the same
department and semester but at another level. For example, an undergraduate and a graduate version taught together
(e.g., CS 3780 and CS 5780, Introduction to Machine Learning). They have different `crse_id`s because graduate students
get different requirements and credit, and this shows Cornell's academic intent more clearly instead of having it as one
blanket course. Part of the graduate AI share comes from graduate versions of undergraduate courses rather than separate
graduate courses.
* `has_level_twin` (1 or 0): the class has a same-title "twin" course at another level in the same department and
  semester. We require the same department so that generic titles used by many departments (like "Financial Accounting")
  aren't counted as twins.
* `is_upper_twin_copy` (1 or 0): the class is the higher-numbered copy of a twin pair. Analysis 1's robustness check
  removes these rows, so each twin pair counts at its lowest level.

In [15]:
classes["title_key"] = classes["title"].str.lower().str.replace(r"[^a-z0-9 ]", "", regex=True).str.strip()
twin_groups = classes.groupby(["roster", "department", "title_key"])
classes["has_level_twin"] = ((twin_groups["crse_id"].transform("nunique") > 1) &
                             (twin_groups["course_level"].transform("nunique") > 1)).astype(int)

print(f"Class-semesters with a same-title twin at another level: {classes['has_level_twin'].sum():,} "
      f"({classes['has_level_twin'].mean():.1%})")
print(f"Share of is_ai = 1 class-semesters that are twins: {classes.loc[classes['is_ai'] == 1, 'has_level_twin'].mean():.1%}")
print("\nis_ai = 1 twins by level:")
print(classes[(classes["is_ai"] == 1) & (classes["has_level_twin"] == 1)]["course_level"].value_counts().to_string())

# Robustness check: keep only the lowest level member of each twin group.
lowest = classes.groupby(["roster", "department", "title_key"])["catalog_nbr"].transform("min")
classes["is_upper_twin_copy"] = ((classes["has_level_twin"] == 1) & (classes["catalog_nbr"] != lowest)).astype(int)
print(f"\nGraduate/upper copies that the robustness check will drop: {classes['is_upper_twin_copy'].sum():,}")

example = classes[(classes["roster"] == "FA25") & (classes["is_ai"] == 1) & (classes["has_level_twin"] == 1)]
example.sort_values(["title_key", "catalog_nbr"])[["subject", "catalog_nbr", "title", "course_level"]].head(10)

Class-semesters with a same-title twin at another level: 14,815 (19.7%)
Share of is_ai = 1 class-semesters that are twins: 36.0%

is_ai = 1 twins by level:
course_level
Grad     249
Upper    246
Intro      3

Graduate/upper copies that the robustness check will drop: 7,482


,subject,catalog_nbr,title,course_level
84425,INFO,1170,AI in Organizations,Intro
84456,INFO,5170,AI in Organizations,Grad
85369,PHYS,4480,Computational Physics,Upper
85383,PHYS,7680,Computational Physics,Grad
85900,STSCI,3740,Data Mining and Machine Learning,Upper
85926,STSCI,5740,Data Mining and Machine Learning,Grad
82808,BEE,4310,Environmental Statistics and Learning,Upper
82827,BEE,6310,Environmental Statistics and Learning,Grad
83378,CS,3700,Foundations of AI Reasoning and Decision-Making,Upper
83415,CS,5700,Foundations of AI Reasoning and Decision-Making,Grad


#### Step 13: A one row per course history table
Our tables so far have one row per class per semester, so a course offered 20 times appears 20 times. To ask when AI
courses first appeared, we need one row per course. SQL's `GROUP BY` collapses all semesters of a course into a single
row, and a subquery (the `SELECT` inside the `LEFT JOIN`) finds the first semester each course was AI-flagged.

`course_history` (one row per `crse_id`) has:
* `first_year`, `last_year`, `semesters_offered`: when the course ran.
* `subject`, `catalog_nbr`, `title`: the course's most recent listing, so every row says which course it is (e.g., Machine
  Learning appears as CS 3780).
* `n_numbers`, `n_titles`: how many numbers and titles it had. More than one number means the course was renumbered, or
  its main listing moved to another department. More than one title means it was renamed.
* `discipline`: the discipline of the most recent listing.
* `ever_ai`, and for AI courses `first_ai_strm` (term code), `first_ai_year`, `first_ai_level`, `first_ai_subject`,
  `first_ai_nbr`: when and where it first counted as AI (Machine Learning first counted as AI as CS 4780).
* `how_became_ai`, which sorts every AI course into one of three groups:
  * **AI already in 2014–15:** AI at the start of our data.
  * **New course, AI from the start:** first offered and first flagged as an AI-related course in the same year.
  * **Existing course, AI added later:** offered before, then its description was rewritten to include AI. These can show
    us more trends.

This tells us whether AI enters the curriculum through new courses or by changing existing ones, letting us be more
thorough.

In [16]:
# Build a course-history table with SQL: one row per course (crse_id), however many semesters it ran.
course_history = duckdb.sql("""
    SELECT c.crse_id,
           MIN(c.academic_year)            AS first_year,
           MAX(c.academic_year)            AS last_year,
           COUNT(*)                        AS semesters_offered,
           COUNT(DISTINCT c.catalog_nbr)   AS n_numbers,
           COUNT(DISTINCT c.title)         AS n_titles,
           MAX(c.is_ai)                    AS ever_ai,
           MIN(f.first_ai_strm)            AS first_ai_strm
    FROM classes AS c
    LEFT JOIN (SELECT crse_id, MIN(strm) AS first_ai_strm   -- first semester each course was AI-flagged
               FROM classes
               WHERE is_ai = 1
               GROUP BY crse_id) AS f
           ON c.crse_id = f.crse_id
    GROUP BY c.crse_id
    ORDER BY c.crse_id
""").df()

# Name each course by its most recent listing, so every row says which course it is.
latest = (classes.sort_values("strm").drop_duplicates("crse_id", keep="last")
          [["crse_id", "subject", "catalog_nbr", "title", "discipline"]])
course_history = latest.merge(course_history, on="crse_id")

# Add what the course looked like the first semester it was AI-flagged.
first_ai_rows = (classes[classes["is_ai"] == 1].sort_values("strm").drop_duplicates("crse_id")
                 [["crse_id", "academic_year", "course_level", "subject", "catalog_nbr"]]
                 .rename(columns={"academic_year": "first_ai_year", "course_level": "first_ai_level",
                                  "subject": "first_ai_subject", "catalog_nbr": "first_ai_nbr"}))
course_history = course_history.merge(first_ai_rows, on="crse_id", how="left")
course_history["first_ai_nbr"] = course_history["first_ai_nbr"].astype("Int64")

# How did each AI course become AI?
ai_courses = course_history[course_history["ever_ai"] == 1].copy()
ai_courses["how_became_ai"] = np.where(
    ai_courses["first_ai_year"] == "2014–15", "AI already in 2014–15",
    np.where(ai_courses["first_ai_year"] == ai_courses["first_year"],
             "New course, AI from the start", "Existing course, AI added later"))
course_history = course_history.merge(ai_courses[["crse_id", "how_became_ai"]], on="crse_id", how="left")

print(f"Class-semesters: {len(classes):,} -> distinct courses: {len(course_history):,}")
print(f"Distinct courses ever flagged is_ai = 1: {len(ai_courses):,}\n")
how_table = ai_courses.groupby(["first_ai_level", "how_became_ai"]).size().unstack(fill_value=0)
how_table["All"] = how_table.sum(axis=1)
how_table.loc["All"] = how_table.sum()
display(how_table.reindex(["Intro", "Upper", "Grad", "All"]))
ai_courses.sort_values("semesters_offered", ascending=False).head(8)[
    ["subject", "catalog_nbr", "title", "first_ai_year", "first_ai_level", "semesters_offered", "how_became_ai"]]

Class-semesters: 75,082 -> distinct courses: 12,876
Distinct courses ever flagged is_ai = 1: 359



how_became_ai,AI already in 2014–15,"Existing course, AI added later","New course, AI from the start",All
first_ai_level,,,,
Intro,5,5,9,19
Upper,13,20,67,100
Grad,19,50,171,240
All,37,75,247,359


,subject,catalog_nbr,title,first_ai_year,first_ai_level,semesters_offered,how_became_ai
10802,CS,7790,Seminar in Artificial Intelligence,2014–15,Grad,25,AI already in 2014–15
10776,CS,3700,Foundations of AI Reasoning and Decision-Making,2014–15,Upper,23,AI already in 2014–15
10799,CS,7690,Computer Graphics and Vision Seminar,2015–16,Grad,23,"Existing course, AI added later"
10778,CS,4701,Practicum in Artificial Intelligence,2014–15,Upper,23,AI already in 2014–15
10783,CS,4820,Introduction to Analysis of Algorithms,2014–15,Upper,22,AI already in 2014–15
10790,CS,5780,Introduction to Machine Learning,2014–15,Grad,20,AI already in 2014–15
11211,INFO,4940,Special Topics in Information Science,2024–25,Upper,20,"Existing course, AI added later"
10780,CS,3780,Introduction to Machine Learning,2014–15,Upper,19,AI already in 2014–15


#### Step 14: Revise the AI flags
We count which keyword triggered each AI indicator flag, and then list flagged courses outside Computing and Engineering &
Math, where a match is most likely to be a false positive. Examples we found:
* Networks (CS 2850 / ECON 2040 / INFO 2040 / SOC 2090) is flagged only because "neural networks" appears in a list of
  topics.
* Introduction to Analysis of Algorithms (CS 4820) lists "artificial intelligence" as one example application of
  algorithms but not an actually taught topic about AI.
* Hollywood Biology (PLBIO 2490), an intro course, lists artificial intelligence among the science topics shown in films.

However, courses such as PSYCH 3020 Methods in Neuroscience do indeed teach "machine learning methods for behavioral
analysis."

In [17]:
print("Which keyword triggered is_ai (distinct courses, first AI semester):")
first_kw = classes[classes["is_ai"] == 1].sort_values("strm").drop_duplicates("crse_id")["ai_keyword"]
print(first_kw.value_counts().to_string())

outside = classes[(classes["is_ai"] == 1) & (~classes["discipline"].isin(["Computing", "Engineering & Math"]))]
print(f"\nis_ai = 1 class-semesters outside Computing and Engineering & Math: {len(outside)}")
(outside.groupby(["subject", "catalog_nbr", "title", "ai_keyword"]).size()
        .rename("semesters").reset_index().sort_values("semesters", ascending=False).head(20))

Which keyword triggered is_ai (distinct courses, first AI semester):
ai_keyword
machine learning               181
artificial intelligence         93
neural network                  17
computer vision                 16
reinforcement learning          14
generative ai                   12
deep learning                   12
natural language processing      8
large language model             6

is_ai = 1 class-semesters outside Computing and Engineering & Math: 380


,subject,catalog_nbr,title,ai_keyword,semesters
108,NBA,6145,AI Strategy and Applications,artificial intelligence,13
45,ECON,2040,Networks,neural network,12
129,PADM,5472,Leveraging Information Technology for Public and Nonprofit Management,artificial intelligence,8
80,ILRLR,4066,Technological Change at Work,machine learning,6
137,PLBIO,6000,Concepts and Techniques in Computational Biology,machine learning,6
136,PLBIO,4000,Concepts and Techniques in Computational Biology,machine learning,6
135,PLBIO,2490,Hollywood Biology: Science in Cinema,artificial intelligence,6
82,ILRLR,6066,Technological Change at Work,machine learning,5
25,BIONB,3300,Introduction to Computational Neuroscience,neural network,5
103,NBA,4920,Machine Learning Applications in Business,machine learning,5


#### Step 14b (new in Phase III): Make the hand-labeling sheets
Our Phase II reviewer asked us to write a clear rule for courses that **teach** AI, **discuss** AI, or only **mention**
AI, and to check our flags by hand, starting with the intro courses. So we check all 19 intro courses and a random
sample of 30 of the other 340 flagged courses, using each course's description from the first semester it was flagged,
since that is the text that made it count.

| Label | Rule | Examples |
|---|---|---|
| **TEACH** | Students learn, build, or use AI/ML methods in the course. | CS 3780 Introduction to Machine Learning; PSYCH 3020 Methods in Neuroscience |
| **DISCUSS** | AI is a main topic of the course, but it is studied without the technical side (policy, ethics, business, work). | PUBPOL 3520 Economic and Policy Implications of Artificial Intelligence; NBA 6145 AI Strategy and Applications |
| **MENTION** | AI only shows up as an example, an application, or one item in a list. | CS 4820 Introduction to Analysis of Algorithms; PLBIO 2490 Hollywood Biology; Networks (CS 2850 / ECON 2040) |

If we can't decide, we delete the AI words from the description: if the course still sounds the same, it is MENTION.

* All three of us label the 19 intro courses without looking at each other's answers, because a few mistakes there could
  change Analysis 1, and labeling them three times lets us report how often we agree.
* We pick 30 of the other flagged courses at random and split them between us (10 each). Picking at random means the
  share that holds up in our sample is a fair estimate for all of the other flagged courses.
* A second sheet has 30 courses that were **not** flagged but have AI-sounding titles (data, learning, algorithm, ...),
  also 10 each, to see if our keywords missed real AI courses. We picked these on purpose, so they can show that misses
  happen, but not how often.

We fill in the `label` column in Google Sheets, write the agreed label in a `resolved` column when we disagree, and
download the sheets as `ai_label_sheet_labeled.csv` and `fn_label_sheet_labeled.csv` into this folder.

In [18]:
# Step 14b: build the hand-labeling sheets (TA feedback on Phase II, question 2).
# Sheet 1: all 19 intro AI courses (labeled by all three of us) + a random sample of 30 other AI courses.
# Sheet 2: a sample of UNflagged courses with AI-adjacent titles, to look for courses our keywords missed.
# The sheets are only written if they don't exist yet, so re-running this cell never erases finished labels.
import os

LABELERS = ["Aiden", "Anthony", "Youssef"]
N_OTHER = 30            # random sample of the other (non-intro) flagged courses, 10 per person
N_FALSE_NEG = 30        # unflagged candidates to check for missed AI courses, 10 per person
SHEET_FLAGGED = "ai_label_sheet.csv"
SHEET_UNFLAGGED = "fn_label_sheet.csv"

# Text of each course in the FIRST semester it was flagged (that's the text that made it count as AI).
first_flag = (classes[classes["is_ai"] == 1].sort_values("strm").drop_duplicates("crse_id")
              [["crse_id", "roster", "subject", "catalog_nbr", "title", "description",
                "ai_keyword", "course_level", "discipline"]])

# Did the course's description change while it was flagged? If yes, the label might not fit every semester.
n_desc = classes[classes["is_ai"] == 1].groupby("crse_id")["description"].nunique()
first_flag["desc_changed"] = first_flag["crse_id"].map(n_desc).gt(1).astype(int)

def snippet(row, width=110):
    """The words around the matched keyword, so labelers can judge it in context."""
    text = f"{row['title']}. {row['description']}"
    pos = text.lower().find(str(row["ai_keyword"]))
    if pos < 0:
        return text[:2 * width]
    return "..." + text[max(0, pos - width): pos + width] + "..."

first_flag["context"] = first_flag.apply(snippet, axis=1)

# Who labels what: all three of us label every intro course.
# A random sample of the other flagged courses is split between us
np.random.seed(2950)
intro_ids = first_flag.loc[first_flag["course_level"] == "Intro", "crse_id"].tolist()
other_all = first_flag.loc[first_flag["course_level"] != "Intro", "crse_id"].to_numpy()
other_ids = np.random.choice(other_all, N_OTHER, replace=False)
assign = [(c, name) for c in intro_ids for name in LABELERS]
assign += [(c, LABELERS[k % 3]) for k, c in enumerate(other_ids)]
sheet = pd.DataFrame(assign, columns=["crse_id", "labeler"]).merge(first_flag, on="crse_id")
sheet["label"] = ""        # TEACH / DISCUSS / MENTION labels
sheet["notes"] = ""
sheet = sheet.sort_values(["labeler", "course_level", "subject", "catalog_nbr"])

# Unflagged candidates: titles that sound AI-similar but contain none of our keywords.
adjacent = (r"(?i)\b(?:data|learning|intelligen\w*|algorithm\w*|computational|predict\w*|automat\w*|robot\w*|"
            r"statistical|neural|cognitive)\b")
unflagged = classes[classes["is_ai"] == 0].sort_values("strm").drop_duplicates("crse_id", keep="last")
unflagged = unflagged[unflagged["title"].str.contains(adjacent, regex=True)
                      & ~unflagged["crse_id"].isin(first_flag["crse_id"])]
fn = unflagged.sample(min(N_FALSE_NEG, len(unflagged)), random_state=2950)[
    ["crse_id", "roster", "subject", "catalog_nbr", "title", "description", "course_level", "discipline"]].copy()
fn["labeler"] = [LABELERS[k % 3] for k in range(len(fn))]
fn["label"] = ""           # TEACH / DISCUSS / NONE labels
fn["notes"] = ""

for path, table in [(SHEET_FLAGGED, sheet), (SHEET_UNFLAGGED, fn)]:
    if os.path.exists(path):
        print(f"{path} already exists, not overwritten")
    else:
        table.to_csv(path, index=False)
        print(f"Wrote {path}: {len(table)} rows")

print(f"\nFlagged courses to label: {len(intro_ids)} intro (labeled by all three) + {len(other_ids)} random others, "
      f"out of {first_flag['crse_id'].nunique()} flagged courses")
print("Rows per labeler (flagged sheet):", sheet["labeler"].value_counts().to_dict())
print(f"Flagged courses whose description changed while flagged: {first_flag['desc_changed'].sum()}")

Wrote ai_label_sheet.csv: 87 rows
Wrote fn_label_sheet.csv: 30 rows

Flagged courses to label: 19 intro (labeled by all three) + 30 random others, out of 359 flagged courses
Rows per labeler (flagged sheet): {'Aiden': 29, 'Anthony': 29, 'Youssef': 29}
Flagged courses whose description changed while flagged: 115


#### Step 14c (new in Phase III): Add the hand labels to our tables
Each course gets the label most of its labelers chose (ties use the `resolved` column). We keep `is_ai` exactly as it is,
because it is the measure we preregistered, and add two new versions for our robustness checks:
* `is_ai_reviewed`: flagged by our keywords **and** labeled TEACH or DISCUSS (this removes the MENTION false positives)
* `is_ai_teach`: flagged **and** labeled TEACH (the strictest version)

Flagged courses outside our sample keep their keyword flag, so the corrected flags change only the courses we checked
(every intro course, and the 30 sampled others). If the labeled sheets aren't in the folder yet, this step does
nothing.

In [19]:
# Step 14c: merge the finished labels back in and build the correct AI flags
LABELED_FLAGGED = "ai_label_sheet_labeled.csv"     # the filled-in sheet downloaded from Google Sheets where we manually organized
LABELED_UNFLAGGED = "fn_label_sheet_labeled.csv"
VALID = {"TEACH", "DISCUSS", "MENTION"}

if os.path.exists(LABELED_FLAGGED):
    lab = pd.read_csv(LABELED_FLAGGED, keep_default_na=False)   # keep_default_na=False: read the word "None" as text, not as missing
    lab["label"] = lab["label"].astype(str).str.strip().str.upper()
    bad = lab[~lab["label"].isin(VALID)]
    print(f"Rows without a valid label (blank or typo): {len(bad)}")
    lab = lab[lab["label"].isin(VALID)]

    # Agreement: among courses labeled by 2 or more people, how often did everyone pick the same label.
    # All of this is a precaution since we did our own parts without looking at what the others put down just to keep it independent
    per_course = lab.groupby("crse_id")["label"].agg(n_labelers="size", n_distinct="nunique")
    multi = per_course[per_course["n_labelers"] > 1]
    print(f"Courses labeled by 2+ people: {len(multi)}; all labelers agreed on {(multi['n_distinct'] == 1).mean():.0%}")

    # A tie is unresolved until the group writes the agreed label in 'resolved'.
    def final_label(labels):
        counts = labels.value_counts()
        return counts.index[0] if (len(counts) == 1 or counts.iloc[0] > counts.iloc[1]) else "UNRESOLVED"
    final = lab.groupby("crse_id")["label"].apply(final_label).rename("final_label").to_frame()
    if "resolved" in lab.columns:
        lab["resolved"] = lab["resolved"].astype(str).str.strip().str.upper()   # blank cells are ignored below
        fixes = lab[lab["resolved"].isin(VALID)].groupby("crse_id")["resolved"].first()
        final.loc[fixes.index, "final_label"] = fixes
    final = final.join(per_course)
    final["agreed"] = (final["n_distinct"] == 1).astype(int)
    print(f"Unresolved ties: {(final['final_label'] == 'UNRESOLVED').sum()} ")

    flagged_ids = set(classes.loc[classes["is_ai"] == 1, "crse_id"])
    print(f"Share of flagged courses with a final label: {len(set(final.index) & flagged_ids) / len(flagged_ids):.0%}")

    for t in (classes, listings):
        t["ai_label"] = np.where(t["is_ai"] == 1, t["crse_id"].map(final["final_label"]), "NONE")
        # Unlabeled or unresolved flagged courses keep the keyword flag
        unknown = t["ai_label"].isna() | (t["ai_label"] == "UNRESOLVED")
        t["is_ai_reviewed"] = ((t["is_ai"] == 1) & (t["ai_label"].isin(["TEACH", "DISCUSS"]) | unknown)).astype(int)
        t["is_ai_teach"] = ((t["is_ai"] == 1) & ((t["ai_label"] == "TEACH") | unknown)).astype(int)

    final.reset_index().to_csv("ai_labels_final.csv", index=False)
    print("\nFinal labels (distinct courses):", final["final_label"].value_counts().to_dict())
    print(f"is_ai class-semesters: {classes['is_ai'].sum():,} -> is_ai_reviewed: {classes['is_ai_reviewed'].sum():,}"
          f" -> is_ai_teach: {classes['is_ai_teach'].sum():,}")
else:
    print(f"{LABELED_FLAGGED} not found: labels not merged yet, so the corrected-flag columns are not created.")

if os.path.exists(LABELED_UNFLAGGED):
    fn_lab = pd.read_csv(LABELED_UNFLAGGED, keep_default_na=False)
    fn_lab["label"] = fn_lab["label"].astype(str).str.strip().str.upper()
    fn_lab = fn_lab[fn_lab["label"].isin({"TEACH", "DISCUSS", "NONE"})]
    fn_lab.to_csv("fn_labels_final.csv", index=False)
    print(f"\nUnflagged candidates checked: {len(fn_lab)}; judged AI after all: "
          f"{fn_lab['label'].isin(['TEACH', 'DISCUSS']).sum()}")

Rows without a valid label (blank or typo): 0
Courses labeled by 2+ people: 19; all labelers agreed on 32%
Unresolved ties: 0 
Share of flagged courses with a final label: 14%

Final labels (distinct courses): {'TEACH': 23, 'DISCUSS': 16, 'MENTION': 10}
is_ai class-semesters: 1,382 -> is_ai_reviewed: 1,319 -> is_ai_teach: 1,299

Unflagged candidates checked: 30; judged AI after all: 2


#### Step 15: Clean Google Trends
Now we go to Google Trends. Google Trends gives one score per month, but our courses are per semester.
1. **Drop incomplete months:** October 2026 is marked `isPartial = True` (the month was in progress when we scraped the data set).
2. **Convert months to semesters:** Spring = January–May, Fall = August–December. June and July are dropped. Fall 2026 only has August and September.
3. **Combine the terms:** We average the three search terms into a combined one per month, then average the months in each semester.
4. **Rescale** so the highest semester in our window equals 100 (`trends_index`). It makes numbers easier to read.
5. **Lag by one semester** (`trends_lag1`). Departments plan courses at least a semester ahead, so a Fall course can only be planned according to interest from the previous Spring.

In [20]:
trends = trends_raw.copy()
trends["date"] = pd.to_datetime(trends["date"])
print("Incomplete months dropped:", int(trends["isPartial"].sum()))
trends = trends[~trends["isPartial"]]

SEARCH_TERMS = ["machine learning", "artificial intelligence", "neural network"]
trends["composite"] = trends[SEARCH_TERMS].mean(axis=1)
trends["year"] = trends["date"].dt.year
month = trends["date"].dt.month
trends["semester"] = np.where(month <= 5, "Spring", np.where(month >= 8, "Fall", "Summer"))
trends = trends[trends["semester"] != "Summer"]

sem_trends = trends.groupby(["year", "semester"], as_index=False)["composite"].mean()
sem_trends["order"] = sem_trends["year"] * 2 + (sem_trends["semester"] == "Fall")
sem_trends = sem_trends.sort_values("order").reset_index(drop=True)
sem_trends["trends_index"] = 100 * sem_trends["composite"] / sem_trends["composite"].max()
# this is so each semester gets the previous semester's score
sem_trends["trends_lag1"] = sem_trends["trends_index"].shift(1)
sem_trends = sem_trends.drop(columns=["order", "composite"])
sem_trends = sem_trends[(sem_trends["year"] > 2014) | (sem_trends["semester"] == "Fall")]

print(f"Semesters with a Trends value: {len(sem_trends)}")
print("Fall 2026 uses only August and September (October is incomplete).")
sem_trends.round(1).set_index(["year", "semester"])

Incomplete months dropped: 1
Semesters with a Trends value: 25
Fall 2026 uses only August and September (October is incomplete).


trends_index  trends_lag1
year semester                           
2014 Fall               6.8          5.8
2015 Spring             8.2          6.8
     Fall               9.3          8.2
2016 Spring            10.5          9.3
     Fall              12.4         10.5
2017 Spring            17.0         12.4
     Fall              20.2         17.0
2018 Spring            21.8         20.2
     Fall              19.8         21.8
2019 Spring            21.1         19.8
     Fall              19.4         21.1
2020 Spring            18.5         19.4
     Fall              16.6         18.5
2021 Spring            15.3         16.6
     Fall              16.3         15.3
2022 Spring            21.0         16.3
     Fall              21.3         21.0
2023 Spring            26.9         21.3
     Fall              24.9         26.9
2024 Spring            27.1         24.9
     Fall              25.3         27.1
2025 Spring            29.4         25.3
     Fall              47.0         29.4
2026 Spring           100.0         47.0
     Fall              43.7        100.0

#### Step 15b (new in Phase III): Save the monthly Google Trends values
Our reviewer suggested plotting the three search terms separately by month to see which one caused the Spring 2026 jump.
We save the monthly values here so the report can make that plot without doing any cleaning.

In [21]:
# Save the monthly Google Trends values (incomplete months removed) for the report's spike check.
trends_monthly = trends_raw.copy()
trends_monthly["date"] = pd.to_datetime(trends_monthly["date"])
trends_monthly = trends_monthly[~trends_monthly["isPartial"].astype(bool)].drop(columns="isPartial")
trends_monthly.to_csv("trends_monthly_clean.csv", index=False)
print(f"Saved trends_monthly_clean.csv: {len(trends_monthly)} months, "
      f"{trends_monthly['date'].min():%b %Y} to {trends_monthly['date'].max():%b %Y}")

Saved trends_monthly_clean.csv: 153 months, Jan 2014 to Sep 2026


#### Step 16: Merge everything in tables
We attached each semester's `trends_index` and `trends_lag1` to each class. Then we built a table for EDA:

`subj_sem` (one row per department for each semester):
* `ai_courses`: listings with `is_ai` = 1. `ai_mentions`: listings with `mentions_ai` = 1
* `total_courses`: all listings that semester
* `crosslisted_ai`, `own_ai_courses`: AI listings that are and aren't cross-listed
* `log_ai_courses` = ln(`ai_courses` + 1). Our Phase II EDA (Plot 4) used it to show why we model individual listings
  instead of department counts.

This cell also saves the four analysis-ready tables (our Phase III report imports `classes`, `listings`, and
`course_history`), which is crucial.

In [22]:
classes = classes.merge(sem_trends, on=["year", "semester"], how="left", validate="many_to_one")
listings = listings.merge(sem_trends, on=["year", "semester"], how="left", validate="many_to_one")
assert classes["trends_lag1"].notna().all(), "every semester needs a lagged Trends value"

# EDA table: one row per department per semester.
listings["crosslisted_ai"] = listings["is_crosslisted"] * listings["is_ai"]
subj_sem = (listings
    .groupby(["roster", "strm", "year", "semester", "term_index", "years_since_start", "academic_year",
              "department", "discipline", "is_computing", "is_computing_broad"], as_index=False)
    .agg(ai_courses=("is_ai", "sum"),
         ai_mentions=("mentions_ai", "sum"),
         total_courses=("is_ai", "size"),
         crosslisted_ai=("crosslisted_ai", "sum"),
         trends_lag1=("trends_lag1", "first")))
subj_sem["own_ai_courses"] = subj_sem["ai_courses"] - subj_sem["crosslisted_ai"]
subj_sem["log_ai_courses"] = np.log(subj_sem["ai_courses"] + 1)

classes.drop(columns="title_key").to_csv("classes_analysis_ready.csv", index=False)
listings.to_csv("listings_analysis_ready.csv", index=False)
subj_sem.to_csv("subject_semester_analysis_ready.csv", index=False)
course_history.to_csv("course_history.csv", index=False)

print(f"classes:        {classes.shape} -> Analysis 1 (one row per class per semester)")
print(f"listings:       {listings.shape} -> Analysis 2 (one row per department listing per semester)")
print(f"subj_sem:       {subj_sem.shape} -> EDA only ({subj_sem['department'].nunique()} departments x "
      f"{subj_sem['roster'].nunique()} semesters, minus semesters a department offered nothing)")
print(f"course_history: {course_history.shape} -> EDA (one row per course across all semesters)")

classes:        (75082, 35) -> Analysis 1 (one row per class per semester)
listings:       (94190, 33) -> Analysis 2 (one row per department listing per semester)
subj_sem:       (3942, 18) -> EDA only (193 departments x 25 semesters, minus semesters a department offered nothing)
course_history: (12876, 17) -> EDA (one row per course across all semesters)
